# Modelos clásicos y cuánticos para la predicción de la estabilidad proteica

**Marco Antonio Roca Rodríguez** · Trabajo Fin de Máster · Máster Universitario en Inteligencia Artificial · Universidad Alfonso X el Sabio (UAX)

<a href="https://colab.research.google.com/github/marrocrod/TFM_UAX/blob/main/notebook/estabilidad_proteica.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"></a>

---

## Objetivo

Predecir el cambio de energía libre de plegamiento $\Delta\Delta G$ (kcal/mol) que provoca una **mutación puntual** en una proteína, y comparar tres modelos sobre las mismas mutaciones y con el mismo protocolo:

| Modelo | Tipo | Implementación |
|---|---|---|
| **XGBoost** | árboles de decisión potenciados | `xgboost` |
| **FCNN** | red neuronal densa | TensorFlow / Keras |
| **QNN** | red híbrida: circuito cuántico variacional de 2 qubits + red clásica | PyTorch (simulación exacta, verificada con Qiskit) |

## Mapa del notebook

| Parte | Bloques | Contenido |
|---|---|---|
| I | 1 – 2 | Entorno y configuración |
| II | 3 – 4 | Datos: S2648, estructuras y localización de las mutaciones |
| III | 5 – 7 | Características: descriptores, estructura 3D, ProtT5 y matrices finales |
| IV | 8 | Protocolo de evaluación |
| V | 9 | XGBoost |
| VI | 10 | FCNN |
| VII | 11 – 13 | QNN: simulador, red híbrida y ablación |
| VIII | 14 | Comparación y resultados |

## Qué cambia en esta versión

Esta versión parte de la versión corregida (los 13 errores descritos en [`docs/correcciones.md`](../docs/correcciones.md)) y añade las mejoras siguientes. Cada una tiene un interruptor en el Bloque 2, para poder medir cuánto aporta.

| # | Mejora | Antes | Ahora |
|---|---|---|---|
| 1 | **Simulación de la QNN** | `EstimatorQNN` evaluaba el circuito muestra a muestra: unos 20 minutos por época | El circuito se simula de forma exacta en PyTorch, por lotes. El Bloque 11 comprueba que da lo mismo que Qiskit |
| 2 | **ProtT5 de la mutación** | Solo el embedding del residuo original: idéntico para todas las mutaciones de una misma posición | Se añade la diferencia entre el embedding de la secuencia mutada y el de la original |
| 3 | **Codificación de residuos** | `ord(letra) − 65`, un número sin significado químico | *One-hot* del residuo original y del mutado; cada vecino se describe con su volumen, hidrofobicidad y flexibilidad |
| 4 | **Estructura 3D** | Las estructuras solo se usaban para extraer la secuencia | Accesibilidad relativa al disolvente, número de contactos y profundidad del residuo mutado |
| 5 | **Condiciones del experimento** | El pH y la temperatura no se usaban | Entran como variables |
| 6 | **Antisimetría** | No se usaba | La mutación inversa tiene $\Delta\Delta G$ de signo contrario: se añade al entrenamiento y la predicción se promedia con la de la inversa |
| 7 | **Ángulos del circuito** | Las componentes PCA entraban sin acotar como ángulos | Se estandarizan y se llevan a $(-\pi, \pi)$ con $2\arctan(z)$ |
| 8 | **Ablación de la QNN** | No existía | La misma red con una capa lineal en lugar del circuito, para medir qué aporta la parte cuántica |

También se han eliminado redundancias: las variables `mutation_type` e índice absoluto (ya contenidas en el *one-hot* y en la posición relativa), el reajuste extra de XGBoost para dibujar curvas, la dependencia de `qiskit-machine-learning` y el bucle de entrenamiento duplicado de la QNN.

---
# Parte I — Entorno y configuración

### 1 · Dependencias

En Google Colab basta con ejecutar esta celda: si el notebook se ha abierto desde GitHub, clona el repositorio para tener los datos, e instala lo que falte. En local, usa `requirements.txt`.

ProtT5-XL necesita GPU para que la extracción de embeddings no tarde horas: en Colab, *Entorno de ejecución → Cambiar tipo de entorno → GPU*.

In [ ]:
import importlib.util, os, subprocess, sys

# On Colab the notebook is opened from GitHub without the rest of the repository:
# clone it once and work from its notebook/ folder so that relative paths resolve.
if "google.colab" in sys.modules and not os.path.exists("../data/s2648.csv"):
    if not os.path.exists("/content/TFM_UAX"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/marrocrod/TFM_UAX", "/content/TFM_UAX"],
                       check=True)
    os.chdir("/content/TFM_UAX/notebook")

REQUIRED = {"Bio": "biopython", "xgboost": "xgboost", "transformers": "transformers",
            "sentencepiece": "sentencepiece", "qiskit": "qiskit>=1.2"}
missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependencias listas" + (f" (instaladas: {', '.join(missing)})" if missing else ""))

### 2 · Configuración

Punto único de ajuste.

- **`RUN_MODE`**: `"smoke"` reduce los datos y las épocas para comprobar en pocos minutos que todo funciona; `"full"` es la ejecución completa.
- **Interruptores de las mejoras**: con todos en `False` se obtiene la línea base (la versión corregida sin mejoras de características). Activarlos de uno en uno permite medir qué aporta cada uno.

In [ ]:
import json, math, os, random, time, warnings
from functools import partial

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# PyTorch must load its compiler stack before TensorFlow loads its own LLVM: otherwise the
# first torch optimizer created after the FCNN can abort the kernel with
# "LLVM ERROR: inconsistency in registered CommandLine options".
import torch
import torch._dynamo  # noqa: F401

# ── Run mode ────────────────────────────────────────────────────────────────
RUN_MODE = "full"                  # "smoke" | "full"
SEED     = 42

# ── Paths (relative to notebook/) ───────────────────────────────────────────
DATA_CSV    = "../data/s2648.csv"
CACHE_DIR   = "../cache"           # PDB files and ProtT5 embeddings (not versioned)
RESULTS_DIR = "../results"
PDB_DIR     = os.path.join(CACHE_DIR, "pdb")
EMB_DIR     = os.path.join(CACHE_DIR, "prott5")
for d in (PDB_DIR, EMB_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)

# ── Feature switches (all False = baseline of the corrected version) ────────
USE_PROTT5_DIFF         = True     # ProtT5(mutant) - ProtT5(wild type) at the mutated position
USE_STRUCTURE           = True     # relative solvent accessibility, contacts, depth
USE_CONDITIONS          = True     # experimental pH and temperature
AUGMENT_ANTISYMMETRY    = True     # add the reverse mutation (-ddG) to the training set
ANTISYMMETRIC_PREDICTION = True    # prediction = (f(forward) - f(reverse)) / 2
QNN_USE_PROTT5          = False    # thesis design: the QNN input has no ProtT5

PROTT5_MODEL = "Rostlab/prot_t5_xl_bfd"
WINDOW       = 7                   # ± sequence neighbours around the mutation

# ── Evaluation protocol ─────────────────────────────────────────────────────
N_FOLDS   = 5                      # grouped cross-validation
TEST_SIZE = 0.20                   # fixed held-out set of unseen protein clusters
GROUP_COL = "CLID"                 # homology cluster of each protein in S2648
VAL_SIZE  = 0.10                   # share of the training clusters used for early stopping

# ── XGBoost ─────────────────────────────────────────────────────────────────
XGB_PARAMS = dict(n_estimators=400, max_depth=5, learning_rate=0.05,
                  subsample=1.0, colsample_bytree=1.0)

# ── FCNN ────────────────────────────────────────────────────────────────────
FCNN_SEEDS         = [42, 7, 1337] # ensemble of three seeds
FCNN_BATCH_SIZE    = 128
FCNN_EPOCHS        = 250
FCNN_PATIENCE      = 16
FCNN_L2            = 1e-5
FCNN_DROPOUT       = 0.40
FCNN_KNN_NEIGHBORS = 28
FCNN_LOSS_WEIGHTS  = (0.50, 0.45, 0.05)   # MSE, (1 - r²), std matching

# ── QNN ─────────────────────────────────────────────────────────────────────
QNN_QUBITS        = 2
QNN_TARGET_DIM    = 16             # PCA components fed to the circuit (8 re-uploading blocks)
QNN_BATCH_SIZE    = 64
QNN_EPOCHS        = 60             # affordable now that the simulation is fast
QNN_PATIENCE      = 8
QNN_LR            = 2e-3
QNN_WD            = 1e-3
QNN_KNN_NEIGHBORS = 15
QNN_LOSS_WEIGHTS  = (0.55, 0.40, 0.05)    # Huber, (1 - r²), std matching

if RUN_MODE == "smoke":
    FCNN_SEEDS, FCNN_EPOCHS = [42], 3
    QNN_EPOCHS, QNN_TARGET_DIM = 2, 4


def set_global_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_global_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Modo: {RUN_MODE.upper()} | semilla {SEED} | dispositivo {device}")
print(f"Mejoras: ProtT5 dif.={USE_PROTT5_DIFF} | estructura={USE_STRUCTURE} | pH/T={USE_CONDITIONS} | "
      f"antisimetría (entrenamiento/predicción)={AUGMENT_ANTISYMMETRY}/{ANTISYMMETRIC_PREDICTION}")

---
# Parte II — Datos

### 3 · Carga de S2648

Cada fila es una mutación: estructura (`PDB`, `CHAIN`), mutación (`MUT`, p. ej. `C30S`), $\Delta\Delta G$ experimental (`DDG`, kcal/mol), condiciones del experimento (`pH`, `T`) y el *cluster* de homología de la proteína (`CLID`).

La posición sigue la **numeración del PDB**, que puede llevar **código de inserción**, como en `L27CN`: residuo L en la posición 27C, mutado a N.

In [ ]:
df = pd.read_csv(DATA_CSV)
parts = df["MUT"].str.extract(r"^([A-Z])(\d+)([A-Z]?)([A-Z])$")
parts.columns = ["wt_res", "position", "icode", "mut_res"]
df = pd.concat([df, parts], axis=1)
df["position"] = df["position"].astype(int)
df["icode"] = df["icode"].fillna("").replace("", " ")     # Biopython uses " " for "no insertion code"

if RUN_MODE == "smoke":
    keep = df[GROUP_COL].drop_duplicates().sample(12, random_state=SEED)
    df = df[df[GROUP_COL].isin(keep)].reset_index(drop=True)

print(f"Mutaciones: {len(df):,} | estructuras: {df['PDB'].nunique()} | clusters: {df[GROUP_COL].nunique()}")
print(f"Con código de inserción: {(df['icode'] != ' ').sum()}")
print(f"ΔΔG: media {df['DDG'].mean():.2f} ± {df['DDG'].std():.2f} kcal/mol "
      f"(rango {df['DDG'].min():.1f} a {df['DDG'].max():.1f})")

### 4 · Estructuras, secuencias y perfil estructural

Para cada cadena se descarga la estructura (formato PDB y, si no existe, mmCIF) y se recorren sus residuos estándar. De ahí salen tres cosas:

- la **secuencia** de la cadena, que es la entrada de ProtT5;
- el **mapa** *(número de residuo del PDB, código de inserción) → índice en la secuencia*, con el que se localiza cada mutación y se comprueba que el aminoácido coincide con el residuo original de S2648;
- un **perfil estructural** por residuo, calculado sobre la cadena aislada y sin hidrógenos ni ligandos:

| Variable | Definición |
|---|---|
| `rsa` | accesibilidad relativa al disolvente: área accesible (Shrake-Rupley) dividida entre el máximo teórico del aminoácido (Tien *et al.*, 2013). ≈ 0 enterrado, ≈ 1 expuesto |
| `contacts_8`, `contacts_12` | número de Cα a menos de 8 y 12 Å |
| `depth` | distancia del Cα al centro de la cadena, dividida entre su radio de giro |

Las mutaciones que no se pueden localizar o cuyo residuo no coincide se descartan, y la celda informa de cuántas son.

In [ ]:
from Bio.PDB import PDBList, PDBParser, MMCIFParser
from Bio.PDB.Atom import Atom
from Bio.PDB.Chain import Chain
from Bio.PDB.Residue import Residue
from Bio.PDB.SASA import ShrakeRupley
from Bio.PDB.Polypeptide import is_aa
from Bio.SeqUtils import seq1

warnings.filterwarnings("ignore", module="Bio")
_pdbl = PDBList(verbose=False)
_sasa = ShrakeRupley()

# Theoretical maximum accessible surface area per residue, in Å² (Tien et al., 2013)
MAX_ASA = {'A': 129, 'R': 274, 'N': 195, 'D': 193, 'C': 167, 'Q': 225, 'E': 223, 'G': 104, 'H': 224, 'I': 197,
           'L': 201, 'K': 236, 'M': 224, 'F': 240, 'P': 159, 'S': 155, 'T': 172, 'W': 285, 'Y': 263, 'V': 174}
STRUCT_NAMES = ["rsa", "contacts_8", "contacts_12", "depth"]


def fetch_structure(pdb_id):
    """Download (or reuse) a structure, trying PDB format first and mmCIF second."""
    pdb_id = pdb_id.lower()
    for fmt, parser in (("pdb", PDBParser(QUIET=True)), ("mmCif", MMCIFParser(QUIET=True))):
        try:
            path = _pdbl.retrieve_pdb_file(pdb_id, pdir=PDB_DIR, file_format=fmt)
            if os.path.exists(path):
                return parser.get_structure(pdb_id, path)
        except Exception:
            continue
    return None


def structural_profile(residues, seq):
    """(L, 4) array with rsa, contacts_8, contacts_12 and depth of every residue."""
    # Clean copy of the chain: standard residues only, no hydrogens, known elements
    clean = Chain("A")
    for i, res in enumerate(residues):
        new = Residue((" ", i, " "), res.get_resname(), "")
        for atom in res:
            el = (atom.element or "C").upper()
            if el in ("H", "D") or new.has_id(atom.get_name()):
                continue
            el = el if el in _sasa.radii_dict else "C"
            new.add(Atom(atom.get_name(), atom.coord, 0.0, 1.0, " ", atom.get_fullname(), 0, element=el))
        clean.add(new)
    _sasa.compute(clean, level="R")
    rsa = np.array([min(1.0, r.sasa / MAX_ASA[a]) for r, a in zip(clean, seq)])

    ca = np.array([r["CA"].coord if "CA" in r else np.mean([a.coord for a in r], axis=0) for r in clean])
    dist = np.linalg.norm(ca[:, None] - ca[None], axis=-1)
    contacts_8 = (dist < 8.0).sum(1) - 1
    contacts_12 = (dist < 12.0).sum(1) - 1
    centred = ca - ca.mean(0)
    depth = np.linalg.norm(centred, axis=1) / max(np.sqrt((centred ** 2).sum(1).mean()), 1e-6)
    return np.stack([rsa, contacts_8, contacts_12, depth], axis=1).astype(np.float32)


def load_chain(structure, chain_id):
    """Sequence, (residue number, insertion code) -> index map and structural profile of a chain."""
    model = next(iter(structure))
    if chain_id not in model:
        return None
    residues = [r for r in model[chain_id] if r.id[0] == " " and is_aa(r, standard=True)]
    if not residues:
        return None
    seq = "".join(seq1(r.get_resname()) for r in residues)
    index = {(r.id[1], r.id[2]): i for i, r in enumerate(residues)}
    return seq, index, structural_profile(residues, seq)


chains = {}                                               # (pdb, chain) -> (sequence, index map, profile)
for pdb_id, chain_id in df[["PDB", "CHAIN"]].drop_duplicates().itertuples(index=False):
    st = fetch_structure(pdb_id)
    entry = load_chain(st, chain_id) if st is not None else None
    if entry is None:
        print(f"  ⚠ {pdb_id} cadena {chain_id}: no disponible")
    else:
        chains[(pdb_id, chain_id)] = entry

# ── Locate every mutation and check the wild-type residue ───────────────────
seq_index, status = [], []
for r in df.itertuples(index=False):
    entry = chains.get((r.PDB, r.CHAIN))
    k = entry[1].get((r.position, r.icode)) if entry else None
    if entry is None:
        seq_index.append(-1); status.append("sin estructura")
    elif k is None:
        seq_index.append(-1); status.append("posición no encontrada")
    else:
        seq_index.append(k); status.append("ok" if entry[0][k] == r.wt_res else "residuo distinto")
df["seq_index"], df["status"] = seq_index, status

print(f"Cadenas cargadas: {len(chains)} de {df[['PDB', 'CHAIN']].drop_duplicates().shape[0]}")
print(df["status"].value_counts().to_string())

# How many mutations the original indexing (position - 1) would have read correctly
ok_old = sum(1 for r in df.itertuples(index=False) if (r.PDB, r.CHAIN) in chains
             and 0 <= r.position - 1 < len(chains[(r.PDB, r.CHAIN)][0])
             and chains[(r.PDB, r.CHAIN)][0][r.position - 1] == r.wt_res)
print(f"\nCon el índice de la versión original (posición − 1) el residuo coincidía en {ok_old} de {len(df)} "
      f"mutaciones ({100 * ok_old / len(df):.1f} %).")

df = df[df["status"] == "ok"].reset_index(drop=True)
print(f"Mutaciones válidas: {len(df):,}")

---
# Parte III — Características

Cada bloque de características se calcula **en los dos sentidos**: para la mutación directa (original → mutado) y para la inversa (mutado → original). La inversa se usa para explotar la antisimetría $\Delta\Delta G_{B\to A} = -\Delta\Delta G_{A\to B}$.

### 5 · Descriptores de la mutación

| Bloque | Dim. | Contenido |
|---|---:|---|
| Residuos | 40 | *one-hot* del residuo de partida (20) y del de llegada (20) |
| Posición | 1 | índice en la secuencia / (longitud − 1) |
| Vecinos | 42 | volumen, hidrofobicidad y flexibilidad de cada vecino a ±7 posiciones (ceros si queda fuera de la cadena) |
| Composición | 20 | recuento de cada aminoácido en esa ventana |
| Fisicoquímica | 3 | Δ volumen, Δ hidrofobicidad y Δ flexibilidad (llegada − partida) |

Los vecinos y la composición son los mismos en los dos sentidos, porque la mutación solo cambia el residuo central. Lo que cambia es el *one-hot* (se intercambia) y el signo de las tres diferencias.

A esto se suman el perfil estructural del residuo (Bloque 4) y las condiciones del experimento (pH y temperatura), que son iguales en los dos sentidos. Para la mutación inversa es una aproximación: se usa la estructura de la proteína original.

In [ ]:
AA20 = "ACDEFGHIKLMNPQRSTVWY"
AA_TO_IDX = {a: i for i, a in enumerate(AA20)}

AA_VOLUME = {'A': 88.6, 'R': 173.4, 'N': 114.1, 'D': 111.1, 'C': 108.5, 'E': 138.4, 'Q': 143.8, 'G': 60.1,
             'H': 153.2, 'I': 166.7, 'L': 166.7, 'K': 168.6, 'M': 162.9, 'F': 189.9, 'P': 112.7, 'S': 89.0,
             'T': 116.1, 'W': 227.8, 'Y': 193.6, 'V': 140.0}
AA_HYDROPHOBICITY = {'A': 0, 'R': 3.71, 'N': 3.47, 'D': 2.95, 'C': 0.49, 'E': 1.64, 'Q': 3.01, 'G': 1.72,
                     'H': 4.76, 'I': -1.56, 'L': -1.81, 'K': 5.39, 'M': -0.76, 'F': -2.2, 'P': -1.52,
                     'S': 1.83, 'T': 1.78, 'W': -0.38, 'Y': -1.09, 'V': -0.78}
AA_FLEXIBILITY = {'A': 1, 'R': 81, 'N': 36, 'D': 18, 'C': 3, 'E': 54, 'Q': 108, 'G': 1, 'H': 36, 'I': 9,
                  'L': 9, 'K': 81, 'M': 27, 'F': 18, 'P': 2, 'S': 3, 'T': 3, 'W': 36, 'Y': 18, 'V': 3}
AA_PROPS = {a: (AA_VOLUME[a], AA_HYDROPHOBICITY[a], AA_FLEXIBILITY[a]) for a in AA20}

OFFSETS = [o for o in range(-WINDOW, WINDOW + 1) if o != 0]
DESC_NAMES = ([f"from_{a}" for a in AA20] + [f"to_{a}" for a in AA20] + ["rel_position"]
              + [f"neigh{o:+d}_{p}" for o in OFFSETS for p in ("vol", "hyd", "flex")]
              + [f"count_{a}" for a in AA20] + ["d_volume", "d_hydrophobicity", "d_flexibility"])


def mutation_descriptors(seq, k, a_from, a_to):
    """Descriptors of the substitution a_from -> a_to at index k of seq."""
    onehot_from, onehot_to = [0.0] * 20, [0.0] * 20
    onehot_from[AA_TO_IDX[a_from]] = 1.0
    onehot_to[AA_TO_IDX[a_to]] = 1.0
    neigh, counts = [], [0.0] * 20
    for off in OFFSETS:
        i = k + off
        if 0 <= i < len(seq):
            neigh += AA_PROPS[seq[i]]
            counts[AA_TO_IDX[seq[i]]] += 1.0
        else:
            neigh += (0.0, 0.0, 0.0)
    delta = [t - f for f, t in zip(AA_PROPS[a_from], AA_PROPS[a_to])]
    return onehot_from + onehot_to + [k / max(1, len(seq) - 1)] + neigh + counts + delta


rows = [(chains[(r.PDB, r.CHAIN)], r.seq_index, r.wt_res, r.mut_res) for r in df.itertuples(index=False)]
D_FWD = np.array([mutation_descriptors(c[0], k, wt, mt) for c, k, wt, mt in rows], dtype=np.float32)
D_REV = np.array([mutation_descriptors(c[0], k, mt, wt) for c, k, wt, mt in rows], dtype=np.float32)
S_STRUCT = np.array([c[2][k] for c, k, _, _ in rows], dtype=np.float32)           # same in both directions
C_COND = df[["pH", "T"]].fillna({"pH": 7.0, "T": 25.0}).to_numpy(np.float32)      # same in both directions

print(f"Descriptores: {D_FWD.shape} | estructura: {S_STRUCT.shape} | condiciones: {C_COND.shape}")
print("Perfil estructural del residuo mutado (media ± desv.): "
      + " | ".join(f"{n} {S_STRUCT[:, i].mean():.2f} ± {S_STRUCT[:, i].std():.2f}" for i, n in enumerate(STRUCT_NAMES)))

### 6 · Embeddings de ProtT5

**ProtT5-XL (BFD)** devuelve, para cada residuo, un vector de 1 024 números que resume su contexto en la secuencia. ProtT5 solo añade un token final (`</s>`), así que el residuo $k$ corresponde a la salida $k$.

Para cada mutación se calculan dos vectores, los dos en la posición mutada:

- $e_{wt}$: embedding con la **secuencia original**;
- $e_{mut}$: embedding con la **secuencia mutada**.

El modelo recibe $e_{wt}$ y la diferencia $e_{mut} - e_{wt}$. El primero describe el entorno del residuo; la segunda, **qué cambia** con esa mutación concreta. Con solo $e_{wt}$, como en la versión anterior, todas las mutaciones de una misma posición tenían el mismo embedding.

Los vectores se guardan por cadena en `cache/prott5/`, así que solo se calculan una vez.

In [ ]:
from transformers import T5Tokenizer, T5EncoderModel

_t5 = {}


def prott5_embed(seq):
    """(L, 1024) per-residue ProtT5 embedding of a sequence."""
    if not _t5:
        _t5["tok"] = T5Tokenizer.from_pretrained(PROTT5_MODEL, do_lower_case=False)
        model = T5EncoderModel.from_pretrained(PROTT5_MODEL).to(device).eval()
        _t5["model"] = model.half() if device == "cuda" else model
    ids = _t5["tok"](" ".join(seq), return_tensors="pt", add_special_tokens=True).to(device)
    with torch.no_grad():
        out = _t5["model"](input_ids=ids["input_ids"], attention_mask=ids["attention_mask"]).last_hidden_state
    assert out.shape[1] == len(seq) + 1, "expected one token per residue plus the final </s>"
    return out[0, :len(seq)].float().cpu().numpy()            # drop </s>; residue k -> row k


def chain_embeddings(pdb_id, chain_id, seq, mutations):
    """{(k, mut_res): (e_wt, e_mut)} for the mutations of one chain, cached on disk."""
    path = os.path.join(EMB_DIR, f"{pdb_id}_{chain_id}.npz")
    keys = sorted(set(mutations))
    if os.path.exists(path):
        z = np.load(path)
        cached = {(int(k), str(m)): i for i, (k, m) in enumerate(zip(z["k"], z["mut"]))}
        if all(key in cached for key in keys):
            return {key: (z["e_wt"][cached[key]], z["e_mut"][cached[key]]) for key in keys}
    wt = prott5_embed(seq)
    e_wt = np.stack([wt[k] for k, _ in keys])
    e_mut = np.stack([prott5_embed(seq[:k] + m + seq[k + 1:])[k] for k, m in keys])
    np.savez(path, k=[k for k, _ in keys], mut=[m for _, m in keys], e_wt=e_wt, e_mut=e_mut)
    return {key: (e_wt[i], e_mut[i]) for i, key in enumerate(keys)}


t0 = time.time()
emb = {}
for (pdb_id, chain_id), g in df.groupby(["PDB", "CHAIN"]):
    muts = list(zip(g["seq_index"], g["mut_res"]))
    for key, pair in chain_embeddings(pdb_id, chain_id, chains[(pdb_id, chain_id)][0], muts).items():
        emb[(pdb_id, chain_id, *key)] = pair

E_WT = np.stack([emb[(r.PDB, r.CHAIN, r.seq_index, r.mut_res)][0] for r in df.itertuples(index=False)]).astype(np.float32)
E_MUT = np.stack([emb[(r.PDB, r.CHAIN, r.seq_index, r.mut_res)][1] for r in df.itertuples(index=False)]).astype(np.float32)
print(f"ProtT5: {E_WT.shape} original y {E_MUT.shape} mutada ({time.time() - t0:.0f} s)")
print(f"Norma media de la diferencia e_mut − e_wt: {np.linalg.norm(E_MUT - E_WT, axis=1).mean():.3f}")

_t5.clear()                                                   # free the GPU for the models below
if device == "cuda":
    torch.cuda.empty_cache()

### 7 · Matrices finales

Se montan cuatro matrices con **las mismas filas**, en el mismo orden:

| Matriz | Para | Contenido |
|---|---|---|
| `X`, `X_REV` | XGBoost y FCNN | descriptores + estructura + condiciones + ProtT5, en sentido directo e inverso |
| `X_QNN`, `X_QNN_REV` | QNN | lo mismo **sin ProtT5**. Es el diseño de la memoria: mantener pequeña la entrada al circuito. `QNN_USE_PROTT5 = True` se lo añade |

`BLOCKS` guarda qué columnas ocupa cada bloque, para poder medir después su importancia.

In [ ]:
def assemble(desc, e_from, e_to, with_prott5=True):
    """Stack the active feature blocks. Returns the matrix and the column slice of each block."""
    parts = [("descriptores", desc)]
    if USE_STRUCTURE:
        parts.append(("estructura", S_STRUCT))
    if USE_CONDITIONS:
        parts.append(("condiciones", C_COND))
    if with_prott5:
        parts.append(("ProtT5 original", e_from))
        if USE_PROTT5_DIFF:
            parts.append(("ProtT5 diferencia", e_to - e_from))
    blocks, start = {}, 0
    for name, a in parts:
        blocks[name] = slice(start, start + a.shape[1])
        start += a.shape[1]
    return np.hstack([a for _, a in parts]).astype(np.float32), blocks


X, BLOCKS = assemble(D_FWD, E_WT, E_MUT)
X_REV, _  = assemble(D_REV, E_MUT, E_WT)                      # reverse mutation: roles swapped
X_QNN, BLOCKS_QNN = assemble(D_FWD, E_WT, E_MUT, with_prott5=QNN_USE_PROTT5)
X_QNN_REV, _      = assemble(D_REV, E_MUT, E_WT, with_prott5=QNN_USE_PROTT5)

y = df["DDG"].to_numpy(np.float32)
groups = df[GROUP_COL].to_numpy()

print(f"X     : {X.shape}  (XGBoost, FCNN)")
for name, sl in BLOCKS.items():
    print(f"   {name:<18} {sl.stop - sl.start:>5}")
print(f"X_QNN : {X_QNN.shape}  (QNN{', con ProtT5' if QNN_USE_PROTT5 else ', sin ProtT5'})")
print(f"y     : {y.shape} | clusters: {len(np.unique(groups))}")

---
# Parte IV — Protocolo de evaluación

### 8 · Particiones, métricas y evaluación

- **Validación cruzada**: `GroupKFold(5)` sobre los *clusters* de proteína. Ninguna proteína aparece a la vez en entrenamiento y en evaluación.
- **Prueba fija**: `GroupShuffleSplit` con un 20 % de los *clusters*.
- **Interfaz común**: cada modelo es una función `fit_predict(X_tr, y_tr, g_tr, X_te)` que hace **todo** su preprocesado dentro (escalado, PCA, kNN, parada temprana), solo con los datos de entrenamiento.
- **Antisimetría**, aplicada igual a todos los modelos dentro de `evaluate()`:
  - *en el entrenamiento*, a cada mutación se le añade su inversa con $-\Delta\Delta G$;
  - *en la predicción*, se predice la mutación directa y la inversa y se devuelve $\tfrac12\,[f(\text{directa}) - f(\text{inversa})]$.

  Solo se evalúa sobre las mutaciones directas, las experimentales.
- **Métricas**: correlación de Pearson (PCC), RMSE y MAE en kcal/mol.

Esta celda define también dos utilidades que comparten la FCNN y la QNN: la meta-característica kNN (predicción de un kNN, calculada *out-of-fold* y por grupos dentro del entrenamiento) y la partición de validación por proteína.

In [ ]:
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.neighbors import KNeighborsRegressor
from scipy.stats import pearsonr

CV_SPLITS = list(GroupKFold(n_splits=N_FOLDS).split(X, y, groups))
TRAIN_IDX, TEST_IDX = next(GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
                           .split(X, y, groups))
assert not set(groups[TRAIN_IDX]) & set(groups[TEST_IDX]), "a protein cluster is on both sides"
print(f"Prueba fija: {len(TEST_IDX)} mutaciones de {len(set(groups[TEST_IDX]))} clusters "
      f"| entrenamiento: {len(TRAIN_IDX)} de {len(set(groups[TRAIN_IDX]))}")


def grouped_holdout(g, size=VAL_SIZE, seed=SEED):
    """Indices (train, val) that keep every protein cluster on one side."""
    return next(GroupShuffleSplit(n_splits=1, test_size=size, random_state=seed).split(np.arange(len(g)), groups=g))


def knn_oof_feature(X_tr, y_tr, g_tr, X_te, n_neighbors):
    """kNN prediction as a meta-feature: out-of-fold (grouped) on train, full fit for test."""
    oof = np.zeros(len(X_tr), dtype=np.float32)
    for tr, va in GroupKFold(n_splits=min(5, len(np.unique(g_tr)))).split(X_tr, y_tr, g_tr):
        knn = KNeighborsRegressor(n_neighbors=min(n_neighbors, len(tr)), weights="distance")
        oof[va] = knn.fit(X_tr[tr], y_tr[tr]).predict(X_tr[va])
    knn = KNeighborsRegressor(n_neighbors=min(n_neighbors, len(X_tr)), weights="distance").fit(X_tr, y_tr)
    return oof.reshape(-1, 1), knn.predict(X_te).astype(np.float32).reshape(-1, 1)


def regression_metrics(y_true, y_pred):
    pcc = pearsonr(y_true, y_pred)[0] if np.std(y_pred) > 0 else 0.0
    mse = mean_squared_error(y_true, y_pred)
    return {"PCC": float(pcc), "RMSE": float(np.sqrt(mse)), "MSE": float(mse),
            "MAE": float(mean_absolute_error(y_true, y_pred))}


def fit_and_predict(fit_predict, F, F_rev, tr, te):
    """Train on tr (plus reverse mutations) and predict the forward mutations of te."""
    X_tr, y_tr, g_tr = F[tr], y[tr], groups[tr]
    if AUGMENT_ANTISYMMETRY:
        X_tr, y_tr, g_tr = np.vstack([X_tr, F_rev[tr]]), np.concatenate([y_tr, -y_tr]), np.concatenate([g_tr, g_tr])
    if not ANTISYMMETRIC_PREDICTION:
        return fit_predict(X_tr, y_tr, g_tr, F[te])
    pred = fit_predict(X_tr, y_tr, g_tr, np.vstack([F[te], F_rev[te]]))
    return 0.5 * (pred[:len(te)] - pred[len(te):])


RESULTS = {}


def evaluate(name, fit_predict, F=None, F_rev=None):
    """Grouped 5-fold CV + fixed held-out set for one model."""
    F, F_rev = (X, X_REV) if F is None else (F, F_rev)
    folds, oof = [], np.zeros_like(y)
    for f, (tr, te) in enumerate(CV_SPLITS, 1):
        t0 = time.time()
        oof[te] = fit_and_predict(fit_predict, F, F_rev, tr, te)
        folds.append(regression_metrics(y[te], oof[te]))
        m = folds[-1]
        print(f"  fold {f}: PCC={m['PCC']:.3f}  RMSE={m['RMSE']:.3f}  MAE={m['MAE']:.3f}  ({time.time() - t0:.0f} s)")
    cv = {k: (float(np.mean([m[k] for m in folds])), float(np.std([m[k] for m in folds]))) for k in folds[0]}
    print(f"  CV  : PCC={cv['PCC'][0]:.3f} ± {cv['PCC'][1]:.3f} | RMSE={cv['RMSE'][0]:.3f} ± {cv['RMSE'][1]:.3f} "
          f"| MAE={cv['MAE'][0]:.3f} ± {cv['MAE'][1]:.3f}")

    pred_test = fit_and_predict(fit_predict, F, F_rev, TRAIN_IDX, TEST_IDX)
    test = regression_metrics(y[TEST_IDX], pred_test)
    print(f"  TEST: PCC={test['PCC']:.3f} | RMSE={test['RMSE']:.3f} | MAE={test['MAE']:.3f}")

    RESULTS[name] = {"cv_folds": folds, "cv": cv, "test": test,
                     "pred_test": pred_test.tolist(), "pred_oof": oof.tolist()}
    return RESULTS[name]

---
# Parte V — XGBoost

### 9 · XGBoost

Árboles de decisión potenciados por gradiente: 400 árboles, profundidad 5 y tasa 0,05. Los árboles no dependen de la escala de las variables, así que no se estandariza.

Al final se muestra la **importancia de cada bloque de características** (suma de la ganancia de sus columnas) en el modelo de la prueba fija. Sirve para ver qué mejoras está usando de verdad el modelo.

In [ ]:
from xgboost import XGBRegressor

XGB_LAST = {}                                                 # last fitted model, for the importance chart


def xgb_fit_predict(X_tr, y_tr, g_tr, X_te):
    model = XGBRegressor(**XGB_PARAMS, random_state=SEED)
    model.fit(X_tr, y_tr, verbose=False)
    XGB_LAST["model"] = model
    return model.predict(X_te)


print("XGBoost")
evaluate("XGBoost", xgb_fit_predict)

# Importance per feature block (model of the fixed split, the last one fitted)
imp = XGB_LAST["model"].feature_importances_
block_imp = {name: float(imp[sl].sum()) for name, sl in BLOCKS.items()}
plt.figure(figsize=(7, 3.2))
plt.barh(list(block_imp)[::-1], [100 * v for v in block_imp.values()][::-1], color="#219BC3")
plt.xlabel("Importancia (% de la ganancia total)"); plt.title("XGBoost — importancia por bloque de características")
plt.grid(axis="x", alpha=0.3); plt.tight_layout(); plt.show()

---
# Parte VI — FCNN

### 10 · Red neuronal densa

- **Red:** `LayerNorm → Dense 320 → Dense 384 → Dense 256 → 1`, con activación *swish*, *dropout* 0,4 y regularización L2.
- **Meta-característica kNN** de 28 vecinos.
- **Pérdida compuesta** sobre $y$ estandarizada: $0{,}50\,\mathrm{MSE} + 0{,}45\,(1-r^2) + 0{,}05\,(\sigma_{\hat y}-1)^2$. El término $(1-r^2)$ empuja a ordenar bien las mutaciones; el último evita que las predicciones se concentren en la media.
- **Optimizador:** Adam con *Cosine Decay Restarts*.
- **Parada temprana** sobre un 10 % de las proteínas de entrenamiento.
- **Ensemble** de 3 semillas.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, callbacks, initializers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import CosineDecayRestarts

tf.get_logger().setLevel("ERROR")


def pcc_tf(y_true, y_pred, eps=1e-8):
    yt = y_true - tf.reduce_mean(y_true)
    yp = y_pred - tf.reduce_mean(y_pred)
    cov = tf.reduce_mean(yt * yp)
    return tf.clip_by_value(cov / (tf.sqrt(tf.reduce_mean(yt ** 2) + eps) * tf.sqrt(tf.reduce_mean(yp ** 2) + eps) + eps),
                            -1.0, 1.0)


def composite_loss_tf(w_mse, w_corr, w_std):
    def loss(y_true, y_pred):
        mse = tf.reduce_mean(tf.square(y_true - y_pred))
        corr = 1.0 - tf.square(pcc_tf(y_true, y_pred))
        std = tf.square(tf.math.reduce_std(y_pred) - 1.0)
        return w_mse * mse + w_corr * corr + w_std * std
    return loss


def build_fcnn(input_dim, seed):
    init = initializers.GlorotUniform(seed=seed)
    x_in = layers.Input(shape=(input_dim,))
    x = layers.LayerNormalization()(x_in)
    for units, drop in ((320, True), (384, True), (256, False)):
        x = layers.Dense(units, kernel_initializer=init, kernel_regularizer=regularizers.l2(FCNN_L2))(x)
        x = layers.Activation("swish")(x)
        if drop:
            x = layers.Dropout(FCNN_DROPOUT, seed=seed)(x)
    model = models.Model(x_in, layers.Dense(1, kernel_initializer=init)(x))
    schedule = CosineDecayRestarts(initial_learning_rate=5e-4, first_decay_steps=400,
                                   t_mul=1.5, m_mul=0.9, alpha=1e-5)
    model.compile(optimizer=Adam(learning_rate=schedule, clipnorm=1.0),
                  loss=composite_loss_tf(*FCNN_LOSS_WEIGHTS), metrics=["mae", pcc_tf])
    return model


FCNN_HISTORIES = []                                           # histories of the last fit, for the chart


def fcnn_fit_predict(X_tr, y_tr, g_tr, X_te):
    # 1) Scaling fitted on the training part only
    sx = StandardScaler().fit(X_tr)
    X_tr, X_te = sx.transform(X_tr), sx.transform(X_te)
    # 2) kNN meta-feature (grouped out-of-fold)
    k_tr, k_te = knn_oof_feature(X_tr, y_tr, g_tr, X_te, FCNN_KNN_NEIGHBORS)
    sk = StandardScaler().fit(k_tr)
    X_tr = np.hstack([X_tr, sk.transform(k_tr)]).astype(np.float32)
    X_te = np.hstack([X_te, sk.transform(k_te)]).astype(np.float32)
    # 3) Standardised target
    sy = StandardScaler().fit(y_tr.reshape(-1, 1))
    y_tr_s = sy.transform(y_tr.reshape(-1, 1)).ravel().astype(np.float32)
    # 4) Validation split by protein cluster, for early stopping
    tr, va = grouped_holdout(g_tr)

    preds = []
    FCNN_HISTORIES.clear()
    for sd in FCNN_SEEDS:
        set_global_seed(sd); tf.random.set_seed(sd)
        model = build_fcnn(X_tr.shape[1], sd)
        h = model.fit(X_tr[tr], y_tr_s[tr], validation_data=(X_tr[va], y_tr_s[va]),
                      epochs=FCNN_EPOCHS, batch_size=FCNN_BATCH_SIZE, shuffle=True, verbose=0,
                      callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=FCNN_PATIENCE,
                                                         restore_best_weights=True),
                                 callbacks.TerminateOnNaN()])
        FCNN_HISTORIES.append(h.history)
        preds.append(sy.inverse_transform(model.predict(X_te, verbose=0).reshape(-1, 1)).ravel())
    return np.mean(preds, axis=0)


print("FCNN")
evaluate("FCNN", fcnn_fit_predict)

# Loss curves of the last fit (fixed split), one pair of lines per seed
plt.figure(figsize=(7, 4.2))
for sd, h in zip(FCNN_SEEDS, FCNN_HISTORIES):
    plt.plot(h["loss"], label=f"entrenamiento (semilla {sd})")
    plt.plot(h["val_loss"], "--", label=f"validación (semilla {sd})")
plt.xlabel("Época"); plt.ylabel("Pérdida compuesta"); plt.title("FCNN — curvas de pérdida (prueba fija)")
plt.grid(alpha=0.3); plt.legend(fontsize=8, ncol=2); plt.tight_layout(); plt.show()

---
# Parte VII — QNN

### 11 · Circuito cuántico y simulador

**Circuito (2 qubits, 8 bloques).** Cada bloque $b$:
1. Codifica dos entradas como ángulos, $R_Y(x_{2b})$ y $R_Y(x_{2b+1})$. Es *data re-uploading*: las 16 entradas entran por pares a lo largo de los 8 bloques.
2. Entrelaza los qubits con una CNOT.
3. Aplica rotaciones entrenables $R_Z(w^z)$ y $R_Y(w^y)$ en cada qubit.

Se miden $\langle Z_0\rangle$, $\langle Z_1\rangle$ y $\langle Z_0Z_1\rangle$. El circuito tiene **32 parámetros**.

**Simulación.** El estado de 2 qubits son 4 amplitudes complejas. En lugar de llamar a un simulador externo muestra a muestra, las puertas se aplican directamente como operaciones de PyTorch sobre todo el lote, y el gradiente sale por *autograd*. El resultado es **exacto**, igual que con `StatevectorEstimator`, y pasa de minutos a milisegundos por lote.

La celda construye el mismo circuito en Qiskit, lo dibuja y **comprueba** que los valores esperados coinciden con los del simulador para entradas y pesos aleatorios.

In [ ]:
import torch.nn as nn


class ReuploadingCircuit(nn.Module):
    """Exact batched state-vector simulation of the data re-uploading circuit.

    forward: (B, blocks * n_qubits) angles -> (B, n_outputs) expectation values
             [<Z_q> for every qubit] + [<Z_q Z_q+1> for neighbouring pairs].
    """
    def __init__(self, n_qubits, n_features):
        super().__init__()
        self.n, self.blocks = n_qubits, math.ceil(n_features / n_qubits)
        self.wz = nn.Parameter(torch.empty(self.blocks, n_qubits).uniform_(-1, 1))
        self.wy = nn.Parameter(torch.empty(self.blocks, n_qubits).uniform_(-1, 1))
        self.n_outputs = n_qubits + (n_qubits - 1)

    # ── gates: the state has shape (B, 2, ..., 2); axis q + 1 is qubit q ─────
    @staticmethod
    def _apply_1q(state, gate, q):
        s = state.movedim(q + 1, -1)
        return torch.einsum("b...j,bij->b...i", s, gate).movedim(-1, q + 1)

    @staticmethod
    def _ry(theta):                                           # theta: (B,)
        c, s = torch.cos(theta / 2), torch.sin(theta / 2)
        return torch.stack([torch.stack([c, -s], -1), torch.stack([s, c], -1)], -2).to(torch.complex64)

    @staticmethod
    def _rz(theta):
        e = torch.exp(-0.5j * theta.to(torch.complex64))
        z = torch.zeros_like(e)
        return torch.stack([torch.stack([e, z], -1), torch.stack([z, e.conj()], -1)], -2)

    @staticmethod
    def _cx(state, control, target):
        s = state.movedim((control + 1, target + 1), (-2, -1))
        s = torch.stack([s[..., 0, :], s[..., 1, :].flip(-1)], dim=-2)   # flip target where control = 1
        return s.movedim((-2, -1), (control + 1, target + 1))

    def forward(self, x):
        B = x.shape[0]
        state = torch.zeros((B,) + (2,) * self.n, dtype=torch.complex64, device=x.device)
        state[(slice(None),) + (0,) * self.n] = 1.0           # |0...0>
        ones = torch.ones(B, device=x.device)
        for b in range(self.blocks):
            for q in range(self.n):
                state = self._apply_1q(state, self._ry(x[:, b * self.n + q]), q)
            for q in range(self.n - 1):
                state = self._cx(state, q, q + 1)
            for q in range(self.n):
                state = self._apply_1q(state, self._rz(self.wz[b, q] * ones), q)
                state = self._apply_1q(state, self._ry(self.wy[b, q] * ones), q)

        probs = state.real ** 2 + state.imag ** 2
        sign = []                                             # eigenvalue of Z_q for every basis state
        for q in range(self.n):
            shape = [1] * (self.n + 1); shape[q + 1] = 2
            sign.append(torch.tensor([1.0, -1.0], device=x.device).view(shape))
        dims = tuple(range(1, self.n + 1))
        z = [(probs * sign[q]).sum(dims) for q in range(self.n)]
        zz = [(probs * sign[q] * sign[q + 1]).sum(dims) for q in range(self.n - 1)]
        return torch.stack(z + zz, dim=1)


# ── Same circuit in Qiskit: drawing and numerical check of the simulator ────
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp, Statevector


def qiskit_circuit(n_qubits, blocks):
    x, wz, wy = (ParameterVector(n, blocks * n_qubits) for n in ("x", "wz", "wy"))
    qc = QuantumCircuit(n_qubits)
    for b in range(blocks):
        for q in range(n_qubits):
            qc.ry(x[b * n_qubits + q], q)
        for q in range(n_qubits - 1):
            qc.cx(q, q + 1)
        for q in range(n_qubits):
            qc.rz(wz[b * n_qubits + q], q)
            qc.ry(wy[b * n_qubits + q], q)
    return qc, x, wz, wy


set_global_seed(SEED)
sim = ReuploadingCircuit(QNN_QUBITS, QNN_TARGET_DIM)
qc, px, pwz, pwy = qiskit_circuit(sim.n, sim.blocks)
x_check = torch.empty(5, sim.blocks * sim.n).uniform_(-math.pi, math.pi)
with torch.no_grad():
    out_torch = sim(x_check).numpy()

observables = ([SparsePauliOp.from_sparse_list([("Z", [q], 1.0)], sim.n) for q in range(sim.n)]
               + [SparsePauliOp.from_sparse_list([("ZZ", [q, q + 1], 1.0)], sim.n) for q in range(sim.n - 1)])
out_qiskit = np.zeros_like(out_torch)
for i in range(len(x_check)):
    values = {**dict(zip(px, x_check[i].tolist())),
              **dict(zip(pwz, sim.wz.detach().flatten().tolist())),
              **dict(zip(pwy, sim.wy.detach().flatten().tolist()))}
    sv = Statevector(qc.assign_parameters(values))
    out_qiskit[i] = [sv.expectation_value(o).real for o in observables]

max_diff = float(np.abs(out_torch - out_qiskit).max())
assert max_diff < 1e-4, f"simulator and Qiskit disagree ({max_diff:.2e})"
print(f"Circuito: {sim.n} qubits, {sim.blocks} bloques, {sim.wz.numel() + sim.wy.numel()} parámetros entrenables, "
      f"{sim.n_outputs} valores medidos")
print(f"Diferencia máxima entre el simulador y Qiskit: {max_diff:.1e}  ✓")
print(qiskit_circuit(sim.n, 2)[0].draw(output="text", fold=120))
print("(se muestran 2 de los", sim.blocks, "bloques)")

### 12 · Red híbrida

```
X_QNN → estandarizado → PCA (16) → estandarizado z ──► ángulos 2·arctan(z) ──► circuito ──► 3 medidas ─┐
                                                 └──► [z ‖ kNN] ─────────────────────────────────────┴─► MLP 64 → 64 → 1
```

- **Entrada.** `X_QNN` se estandariza y se reduce por PCA a 16 componentes. Las componentes se vuelven a estandarizar y se llevan al intervalo $(-\pi, \pi)$ con $2\arctan(z)$, para que cada una sea un ángulo de rotación válido. Antes entraban sin acotar: una componente con valor 9 y otra con valor $9 - 4\pi$ producían la misma rotación.
- **Rama clásica.** Las 16 componentes y la meta-característica kNN.
- **Pérdida.** $0{,}55\,\mathrm{Huber} + 0{,}40\,(1-r^2) + 0{,}05\,(\sigma_{\hat y}-1)^2$, con AdamW y `ReduceLROnPlateau`.
- **Parada temprana** sobre un 10 % de las proteínas de entrenamiento, nunca sobre la prueba.

El argumento `quantum=False` sustituye el circuito por una capa lineal con la misma forma (16 → 3). Es lo que usa la ablación del Bloque 13.

In [ ]:
class HybridRegressor(nn.Module):
    """Quantum branch (or a linear layer of the same shape) concatenated with the classical branch."""
    def __init__(self, n_angles, classical_dim, quantum=True):
        super().__init__()
        if quantum:
            self.q = ReuploadingCircuit(QNN_QUBITS, n_angles)
            q_out = self.q.n_outputs
        else:
            q_out = QNN_QUBITS + (QNN_QUBITS - 1)
            self.q = nn.Linear(n_angles, q_out)
        self.mlp = nn.Sequential(nn.Linear(q_out + classical_dim, 64), nn.GELU(), nn.Dropout(0.20),
                                 nn.Linear(64, 64), nn.GELU(), nn.Dropout(0.10), nn.Linear(64, 1))

    def forward(self, angles, classical):
        return self.mlp(torch.cat([self.q(angles), classical], dim=1)).squeeze(-1)


def pcc_torch(a, b, eps=1e-8):
    a, b = a - a.mean(), b - b.mean()
    return torch.clamp((a * b).mean() / (torch.sqrt((a * a).mean() + eps) * torch.sqrt((b * b).mean() + eps) + eps), -1, 1)


def composite_loss_torch(pred, target):
    w_h, w_c, w_s = QNN_LOSS_WEIGHTS
    return (w_h * nn.functional.huber_loss(pred, target, delta=1.0)
            + w_c * (1.0 - pcc_torch(pred, target) ** 2)
            + w_s * (torch.std(pred) - 1.0) ** 2)


QNN_HISTORIES = {}                                            # validation loss of the last fit of each variant


def qnn_fit_predict(X_tr, y_tr, g_tr, X_te, quantum=True):
    set_global_seed(SEED)
    # 1) Scaling + PCA + component scaling, all fitted on the training part only
    sx = StandardScaler().fit(X_tr)
    pca = PCA(n_components=QNN_TARGET_DIM, random_state=SEED).fit(sx.transform(X_tr))
    sz = StandardScaler().fit(pca.transform(sx.transform(X_tr)))
    Z_tr = sz.transform(pca.transform(sx.transform(X_tr))).astype(np.float32)
    Z_te = sz.transform(pca.transform(sx.transform(X_te))).astype(np.float32)
    # 2) Angles in (-π, π), padded to a whole number of re-uploading blocks
    pad = (-Z_tr.shape[1]) % QNN_QUBITS
    to_angles = lambda Z: np.pad(2.0 * np.arctan(Z), ((0, 0), (0, pad))).astype(np.float32)
    A_tr, A_te = to_angles(Z_tr), to_angles(Z_te)
    # 3) Classical branch: components + grouped out-of-fold kNN meta-feature
    k_tr, k_te = knn_oof_feature(Z_tr, y_tr, g_tr, Z_te, QNN_KNN_NEIGHBORS)
    sk = StandardScaler().fit(k_tr)
    C_tr = np.hstack([Z_tr, sk.transform(k_tr)]).astype(np.float32)
    C_te = np.hstack([Z_te, sk.transform(k_te)]).astype(np.float32)
    # 4) Standardised target and validation split by protein cluster
    sy = StandardScaler().fit(y_tr.reshape(-1, 1))
    y_tr_s = sy.transform(y_tr.reshape(-1, 1)).ravel().astype(np.float32)
    tr, va = grouped_holdout(g_tr)

    t = torch.from_numpy
    A_va, C_va, y_va = t(A_tr[va]), t(C_tr[va]), t(y_tr_s[va])
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(t(A_tr[tr]), t(C_tr[tr]), t(y_tr_s[tr])),
        batch_size=QNN_BATCH_SIZE, shuffle=True, drop_last=len(tr) > QNN_BATCH_SIZE)

    net = HybridRegressor(A_tr.shape[1], C_tr.shape[1], quantum=quantum)
    opt = torch.optim.AdamW(net.parameters(), lr=QNN_LR, weight_decay=QNN_WD)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3, min_lr=1e-5)

    best, bad, best_state, history = float("inf"), 0, None, []
    for epoch in range(QNN_EPOCHS):
        net.train()
        for ab, cb, yb in loader:
            opt.zero_grad()
            composite_loss_torch(net(ab, cb), yb).backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
        net.eval()
        with torch.no_grad():                                 # validation set, never the test set
            val = composite_loss_torch(net(A_va, C_va), y_va).item()
        history.append(val)
        sch.step(val)
        if val + 1e-6 < best:
            best, bad = val, 0
            best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
        else:
            bad += 1
            if bad >= QNN_PATIENCE:
                break
    QNN_HISTORIES["QNN" if quantum else "lineal"] = history
    if best_state is not None:
        net.load_state_dict(best_state)
    net.eval()
    with torch.no_grad():
        pred_s = net(t(A_te), t(C_te)).numpy()
    return sy.inverse_transform(pred_s.reshape(-1, 1)).ravel()


print("QNN")
evaluate("QNN", qnn_fit_predict, X_QNN, X_QNN_REV)

### 13 · Ablación: ¿aporta algo el circuito?

Se repite la evaluación con la misma red, los mismos datos, las mismas particiones y la misma semilla, cambiando **solo** el circuito por una capa `Linear(16 → 3)`:

| | Circuito | Capa lineal |
|---|---|---|
| Parámetros de la rama | 32 | 51 |
| Resto de la red | idéntico | idéntico |

Si las métricas son equivalentes, la parte cuántica no aporta nada medible sobre un componente clásico del mismo tamaño. Si el circuito gana de forma consistente en las 5 particiones, hay indicio de que sí.

In [ ]:
print("QNN — ablación con capa lineal")
evaluate("QNN (lineal)", partial(qnn_fit_predict, quantum=False), X_QNN, X_QNN_REV)

q, l = RESULTS["QNN"], RESULTS["QNN (lineal)"]
diffs = [a["PCC"] - b["PCC"] for a, b in zip(q["cv_folds"], l["cv_folds"])]
print(f"\nPCC en validación cruzada — circuito {q['cv']['PCC'][0]:.3f} ± {q['cv']['PCC'][1]:.3f} "
      f"| lineal {l['cv']['PCC'][0]:.3f} ± {l['cv']['PCC'][1]:.3f}")
print(f"Diferencia por partición (circuito − lineal): {', '.join(f'{d:+.3f}' for d in diffs)} "
      f"| media {np.mean(diffs):+.3f}")
print(f"PCC en la prueba fija — circuito {q['test']['PCC']:.3f} | lineal {l['test']['PCC']:.3f}")

plt.figure(figsize=(6.5, 3.8))
for name, h in QNN_HISTORIES.items():
    plt.plot(h, label=name)
plt.xlabel("Época"); plt.ylabel("Pérdida de validación"); plt.title("QNN — validación (prueba fija)")
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

---
# Parte VIII — Comparación y resultados

### 14 · Tabla, gráficos y registro

Resume los modelos (media ± desviación de la validación cruzada y prueba fija), dibuja la comparación y las predicciones de la prueba fija, y guarda todo en `results/`:

- `resultados.json`: métricas por partición, medias, prueba fija y configuración;
- `predicciones_prueba.csv`: predicciones de cada modelo para cada mutación de la prueba fija;
- `comparacion.png` y `dispersion_prueba.png`.

Después de una ejecución completa, `python scripts/actualizar_resultados.py` lleva estas cifras al README.

In [ ]:
names = list(RESULTS)
summary = pd.DataFrame([{
    "Modelo": n,
    "PCC (CV)": f"{r['cv']['PCC'][0]:.3f} ± {r['cv']['PCC'][1]:.3f}",
    "RMSE (CV)": f"{r['cv']['RMSE'][0]:.3f} ± {r['cv']['RMSE'][1]:.3f}",
    "MAE (CV)": f"{r['cv']['MAE'][0]:.3f} ± {r['cv']['MAE'][1]:.3f}",
    "PCC (prueba)": round(r["test"]["PCC"], 3),
    "RMSE (prueba)": round(r["test"]["RMSE"], 3),
    "MAE (prueba)": round(r["test"]["MAE"], 3)} for n, r in RESULTS.items()]).set_index("Modelo")
try:
    display(summary)                                         # Jupyter / Colab
except NameError:
    print(summary.to_string())

COLORS = {"XGBoost": "#219BC3", "FCNN": "#00566C", "QNN": "#7D3C98", "QNN (lineal)": "#B9A3C9"}
colors = [COLORS.get(n, "gray") for n in names]
xs = np.arange(len(names))

# ── Comparison chart ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ["PCC", "RMSE"]):
    ax.bar(xs - 0.18, [RESULTS[n]["cv"][metric][0] for n in names], 0.34,
           yerr=[RESULTS[n]["cv"][metric][1] for n in names], capsize=4, color=colors,
           label="validación cruzada (media ± desv.)")
    ax.bar(xs + 0.18, [RESULTS[n]["test"][metric] for n in names], 0.34, color=colors, alpha=0.45,
           label="prueba fija")
    ax.set_xticks(xs); ax.set_xticklabels(names); ax.grid(axis="y", alpha=0.3)
    ax.set_title(metric + (" (kcal/mol)" if metric == "RMSE" else ""))
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, "comparacion.png"), dpi=200); plt.show()

# ── Predicted vs experimental on the fixed split ────────────────────────────
y_test = y[TEST_IDX]
lo, hi = float(y.min()) - 0.5, float(y.max()) + 0.5
fig, axes = plt.subplots(1, len(names), figsize=(4.6 * len(names), 4.4), sharex=True, sharey=True)
for ax, n, c in zip(np.atleast_1d(axes), names, colors):
    ax.scatter(y_test, RESULTS[n]["pred_test"], s=14, alpha=0.55, color=c)
    ax.plot([lo, hi], [lo, hi], "--", color="gray", lw=1)
    ax.set_title(f"{n} — PCC {RESULTS[n]['test']['PCC']:.3f} · RMSE {RESULTS[n]['test']['RMSE']:.2f}", fontsize=10)
    ax.set_xlabel("ΔΔG experimental (kcal/mol)"); ax.grid(alpha=0.3)
np.atleast_1d(axes)[0].set_ylabel("ΔΔG predicho (kcal/mol)")
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, "dispersion_prueba.png"), dpi=200); plt.show()

# ── Save results ────────────────────────────────────────────────────────────
pd.DataFrame({"PDB": df["PDB"].values[TEST_IDX], "CHAIN": df["CHAIN"].values[TEST_IDX],
              "MUT": df["MUT"].values[TEST_IDX], "CLID": groups[TEST_IDX], "DDG": y_test,
              **{f"pred_{n}": RESULTS[n]["pred_test"] for n in names}}
             ).to_csv(os.path.join(RESULTS_DIR, "predicciones_prueba.csv"), index=False)

out = {"run_mode": RUN_MODE, "seed": SEED,
       "datos": {"mutaciones": int(len(df)), "clusters": int(len(np.unique(groups))),
                 "prueba_mutaciones": int(len(TEST_IDX)), "prueba_clusters": int(len(set(groups[TEST_IDX]))),
                 "dim_caracteristicas": int(X.shape[1]), "dim_caracteristicas_qnn": int(X_QNN.shape[1]),
                 "bloques": {k: v.stop - v.start for k, v in BLOCKS.items()}},
       "mejoras": {"prott5_diferencia": USE_PROTT5_DIFF, "estructura": USE_STRUCTURE, "condiciones": USE_CONDITIONS,
                   "antisimetria_entrenamiento": AUGMENT_ANTISYMMETRY,
                   "antisimetria_prediccion": ANTISYMMETRIC_PREDICTION, "qnn_usa_prott5": QNN_USE_PROTT5},
       "protocolo": {"cv": f"GroupKFold({N_FOLDS}) por {GROUP_COL}", "prueba": f"GroupShuffleSplit({TEST_SIZE})"},
       "importancia_bloques_xgboost": block_imp,
       "modelos": {n: {k: v for k, v in r.items() if not k.startswith("pred_")} for n, r in RESULTS.items()}}
with open(os.path.join(RESULTS_DIR, "resultados.json"), "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False, indent=2)
print(f"Resultados guardados en {os.path.abspath(RESULTS_DIR)}")